# Divvy Bike-Share Analytics
## Notebook 04 of 06 · Data Quality Assessment

| | |
|---|---|
| **Project** | End-to-end analytics pipeline on Chicago Divvy bike-share trips (January 2024 – September 2026) |
| **CRISP-DM phase** | Data Understanding / Evaluation |
| **Stack** | Python · pandas · NumPy · PostgreSQL (`psycopg2`) · matplotlib |
| **Input** | The loaded `divvy` warehouse: 16,440,836 trips plus four dimension tables |
| **Output** | A scored data-quality report (Markdown, JSON, CSV) and missing-value charts |

### Objective
Answer one question before any analysis: *can the loaded data be trusted?* The audit re-derives every metric from the warehouse itself rather than relying on the ETL pipeline's own counters.

### Approach
1. Pull every table into memory once, so all checks run on one consistent snapshot.
2. Run nine independent checks: duplicate IDs, missing stations, invalid timestamps, negative durations, impossible coordinates, null percentages, cardinality, completeness and consistency.
3. Visualise missingness as a table-by-column heatmap and a row-level presence matrix.
4. Roll the checks into a weighted 0–100 score across five standard dimensions: Uniqueness, Completeness, Validity, Consistency and Accuracy.
5. Publish the findings as Markdown, JSON and CSV from a single source of truth.

### Results at a glance (run of 2026-10-16, 16,440,836 trips)

| # | Check | Result | Verdict |
|---|---|---|---|
| 1 | Duplicate ride IDs | 0 duplicates | PASS |
| 2 | Missing stations | about 20% null start station, about 21% null end station; 0 orphaned keys | **FAIL** |
| 3 | Invalid timestamps | 0 invalid rows | PASS |
| 4 | Negative ride durations | 0 non-positive; 314 stored values differ from the recomputed duration (about 0.002%) | PASS |
| 5 | Impossible coordinates | 0 impossible, 0 at (0, 0); 17,107 null end coordinates reported separately | PASS |
| 6 | Null percentages | `is_round_trip` 31.57%, end station 20.91%, start station 20.07% | no verdict |
| 7 | Cardinality | Every column behaves as its role implies | no verdict |
| 8 | Completeness | 93.17% overall; no calendar gaps | no verdict |
| 9 | Consistency | `day_of_week` and `start_date_key` disagree with `started_at` in about 19.1% of rows; `month_partition` in about 0.6% | **FAIL** |

**Overall quality score: 90.5 / 100 (Grade A)**, with Consistency (61.27) the weakest dimension. The two FAILs are the findings that matter: missing station keys are a genuine completeness gap, and the date-field mismatches are most likely an artifact of the audit reading Chicago-local timestamps in UTC rather than an ETL error. Supporting evidence is given under check 9; the confirming recompute in `America/Chicago` time is still outstanding.

## 1. Setup

pandas and NumPy drive the checks and statistics, `psycopg2` reads the warehouse, and matplotlib draws the charts. The local `config` module keeps credentials out of the notebook.

In [1]:
# Standard library: report output, paths and timestamps
import json
import os
import warnings
from datetime import datetime, timezone

# Analysis and statistics
import numpy as np
import pandas as pd
# PostgreSQL access
import psycopg2
# Charts
import matplotlib.pyplot as plt
# Render the report as formatted Markdown in the notebook
from IPython.display import Markdown, display

# Local module that supplies database credentials
import config

# Silence a known pandas notice about raw DBAPI connections; results are unaffected
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.facecolor"] = "white"

### Configuration

Every threshold lives in one cell so nothing is a magic number inside a check: report folder, an optional row sample for fast iteration (`SAMPLE_ROWS`, left as `None` for a full assessment), geographic bounding boxes, Divvy's launch date, a tolerance for comparing stored and recomputed durations, and the two thresholds that turn a "% of rows affected" figure into a verdict (at least 1% is FAIL, at least 0.1% is WARN).

Defining severity once makes every check consistent and auditable: a reviewer can see what FAIL means before reading any result. The thresholds apply to checks 1–5 and 9; checks 6–8 report raw metrics only. A check can therefore PASS while still surfacing a small real issue, such as the 314 duration mismatches below.

In [2]:
# Report outputs are written to reports/data_quality_reports
PROJECT_FOLDER = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
REPORTS_FOLDER = os.path.join(PROJECT_FOLDER, "reports", "data_quality_reports")
os.makedirs(REPORTS_FOLDER, exist_ok=True)

SCHEMA = "divvy"

# Set an integer to sample fact_trip while iterating; None assesses the full table
SAMPLE_ROWS = None

# Chicago-area bounding box (matches the ETL): separates out-of-service-area outliers from impossible coordinates
CHI_LAT_RANGE = (41.55, 42.15)
CHI_LNG_RANGE = (-88.10, -87.40)

# Physically valid coordinate ranges anywhere on Earth
WORLD_LAT_RANGE = (-90.0, 90.0)
WORLD_LNG_RANGE = (-180.0, 180.0)

# Divvy's launch date: earlier or future timestamps are suspect
PROGRAM_LAUNCH = pd.Timestamp("2013-06-01", tz="UTC")

# Allowed gap between stored and recomputed duration before it counts as a derivation error
DURATION_TOLERANCE_MIN = 0.5

# Verdict thresholds on the share of rows affected, defined once so every check is graded the same way
FAIL_THRESHOLD_PCT = 1.0
WARN_THRESHOLD_PCT = 0.1

# Timestamp the run for the report
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Reports will be written to: {REPORTS_FOLDER}")
print(f"Run timestamp (UTC): {RUN_TIMESTAMP}")

Reports will be written to: C:\Projects\comprehensive_divvy_bike_sharing_analytics\reports\data_quality_reports
Run timestamp (UTC): 2026-11-02T15:08:28+00:00


### Database connection and helpers

`connect_divvy()` opens the connection, `run_query()` returns any query as a DataFrame, and `verdict()` is the single rule that converts a percentage into PASS, WARN or FAIL, so all checks grade themselves identically.

In [3]:
def connect_divvy():
    """Open an autocommit connection to the `divvy_db` warehouse (read-only analysis)."""
    conn = None
    try:
        params = config.config_divvy()
        conn = psycopg2.connect(**params)
        # Autocommit is fine for read-only queries
        conn.autocommit = True
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        return None

def run_query(conn, sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

def verdict(pct_affected: float) -> str:
    """Map the percentage of affected rows to PASS, WARN or FAIL using the configured thresholds."""
    if pct_affected >= FAIL_THRESHOLD_PCT:
        return "FAIL"
    if pct_affected >= WARN_THRESHOLD_PCT:
        return "WARN"
    return "PASS"

## 2. Load data for assessment

All five `divvy` tables are read once and the connection is closed, so every check works from the same point-in-time snapshot and the warehouse is not held open while the checks run. Timestamp columns are explicitly cast to timezone-aware datetimes because `psycopg2` returns plain Python objects.

In [ ]:
# Pull every table once so all checks run on one consistent snapshot
conn = connect_divvy()
if conn is None:
    raise ConnectionError("Could not connect to divvy_db.")

# Optional row cap for quick iteration
_limit_clause = f"LIMIT {SAMPLE_ROWS}" if SAMPLE_ROWS else ""

# Fact table ordered by surrogate key, plus the four dimensions
fact = run_query(conn, f"SELECT * FROM {SCHEMA}.fact_trip ORDER BY trip_key {_limit_clause};")
dim_date = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_date;")
dim_station = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_station;")
dim_ride_type = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_ride_type;")
dim_member_type = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_member_type;")

# Release the connection; all further work happens in pandas
conn.close()

# Ensure timezone-aware datetime dtypes (psycopg2 returns Python datetime objects)
fact["started_at"] = pd.to_datetime(fact["started_at"], utc=True)
fact["ended_at"] = pd.to_datetime(fact["ended_at"], utc=True)
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"])

# Registry of tables used by the null-percentage and report sections
tables = {
    "fact_trip": fact,
    "dim_date": dim_date,
    "dim_station": dim_station,
    "dim_ride_type": dim_ride_type,
    "dim_member_type": dim_member_type,
}

# Sanity check: row and column counts per table
for name, tdf in tables.items():
    print(f"{name:<16} rows={len(tdf):>10,}  cols={tdf.shape[1]}")

## 3. Data quality checks

Each check records its metrics in one shared `findings` dictionary. That dictionary drives the report, the charts and the composite score, while keeping every check self-contained and independently re-runnable.

In [ ]:
# Shared registry: every check records its metrics here, and the score and report are built from it
findings = {}

### Check 1: Duplicate ride IDs

`ride_id` is declared `UNIQUE` in notebook 02, but a constraint can be bypassed by a bulk load, a manual edit or a schema built without it. This check independently counts occurrences of every `ride_id` and shows example rows if any exist.

In [ ]:
# Count occurrences of each ride_id; any count above 1 is a duplicate
ride_id_counts = fact["ride_id"].value_counts()
dup_groups = ride_id_counts[ride_id_counts > 1]
dup_rows = fact[fact["ride_id"].isin(dup_groups.index)]

# Share of the table involved in duplicates
n_dup_rows = len(dup_rows)
pct_dup_rows = round(100 * n_dup_rows / len(fact), 4) if len(fact) else 0.0

# Record metrics and verdict for the report
findings["duplicate_ride_ids"] = {
    "duplicate_ride_id_groups": int(len(dup_groups)),
    "duplicate_rows_total": int(n_dup_rows),
    "pct_rows_affected": pct_dup_rows,
    "verdict": verdict(pct_dup_rows),
}

print(f"Duplicate ride_id groups : {len(dup_groups):,}")
print(f"Rows involved in duplicates: {n_dup_rows:,} ({pct_dup_rows}% of fact_trip)")
# Show example offending rows (none in this run)
display(dup_rows.sort_values('ride_id').head(10)[["trip_key", "ride_id", "started_at", "ended_at"]])

### Check 2: Missing stations

Station problems can arise in three different ways, and each is tested separately: (a) trips whose station key is `NULL`, (b) *orphaned* keys that point to no row in `dim_station`, and (c) dimension rows that are themselves incomplete.

In [ ]:
# NULL station keys (a completeness problem)
null_start_station = fact["start_station_key"].isna().sum()
null_end_station = fact["end_station_key"].isna().sum()

# Orphaned keys: present on the fact table but absent from dim_station (a referential-integrity problem)
valid_station_keys = set(dim_station["station_key"])
orphan_start = fact.loc[fact["start_station_key"].notna() & ~fact["start_station_key"].isin(valid_station_keys)]
orphan_end = fact.loc[fact["end_station_key"].notna() & ~fact["end_station_key"].isin(valid_station_keys)]

# Incomplete dimension rows
station_missing_name = dim_station["station_name"].isna().sum()
station_missing_coords = dim_station[["latitude", "longitude"]].isna().any(axis=1).sum()

# Row-level impact: trips missing either station key
total_missing_station_rows = int(((fact["start_station_key"].isna()) | (fact["end_station_key"].isna())).sum())
pct_missing_station = round(100 * total_missing_station_rows / len(fact), 4) if len(fact) else 0.0

# Record metrics and verdict for the report
findings["missing_stations"] = {
    "null_start_station_key": int(null_start_station),
    "null_end_station_key": int(null_end_station),
    "orphan_start_station_refs": int(len(orphan_start)),
    "orphan_end_station_refs": int(len(orphan_end)),
    "dim_station_missing_name": int(station_missing_name),
    "dim_station_missing_coords": int(station_missing_coords),
    "pct_rows_affected": pct_missing_station,
    "verdict": verdict(pct_missing_station),
}

print(f"NULL start_station_key : {null_start_station:,} ({null_start_station/len(fact):.2%})")
print(f"NULL end_station_key   : {null_end_station:,} ({null_end_station/len(fact):.2%})")
print(f"Orphan start_station_key refs (FK integrity): {len(orphan_start):,}")
print(f"Orphan end_station_key refs (FK integrity)  : {len(orphan_end):,}")
print(f"dim_station rows missing a name  : {station_missing_name:,}")
print(f"dim_station rows missing lat/lng : {station_missing_coords:,}")

**Result: FAIL.** This is the largest quality issue in the warehouse, and it is a completeness problem, not an integrity problem.
- 20.07% of trips (3,299,293) have no start station key and 20.91% (3,437,205) have no end station key.
- There are 0 orphaned keys and `dim_station` has no missing names or coordinates, so the foreign keys are sound; the gaps are trips with no station recorded.
- **Likely cause (unconfirmed):** Divvy leaves station fields blank for trips that did not start or end at a docking station, such as dockless or e-bike rides. Cross-tabulating the null rate against ride type would confirm it.
- **Scale:** 31.57% of trips (about 5.19 million) lack at least one station key. By inclusion and exclusion, roughly 1.55 million lack both, 1.75 million only the start and 1.89 million only the end.
- **Impact:** station-level analysis covers only about 68% of trips (about 80% for either end taken alone) and may skew toward docked ride types.

### Check 3: Invalid timestamps

Tests for missing timestamps, trips that end at or before they start (which the database constraint should already prevent), and timestamps outside Divvy's operating history, either before launch or in the future. The sub-checks are combined into one verdict but reported individually so the root cause stays visible.

In [ ]:
# Missing timestamps
null_started = fact["started_at"].isna().sum()
null_ended = fact["ended_at"].isna().sum()

# Trips that appear to end at or before their start
both_present = fact["started_at"].notna() & fact["ended_at"].notna()
start_ge_end = int((fact.loc[both_present, "started_at"] >= fact.loc[both_present, "ended_at"]).sum())

# Timestamps before Divvy's launch or in the future (the upper bound is the run time)
now_utc = pd.Timestamp.now(tz="UTC")
present_started = fact.loc[fact["started_at"].notna(), "started_at"]
out_of_program_range = int(((present_started < PROGRAM_LAUNCH) | (present_started > now_utc)).sum())

# Combine the sub-checks into one verdict while keeping each count visible
n_invalid_ts = int(null_started + null_ended + start_ge_end + out_of_program_range)
pct_invalid_ts = round(100 * n_invalid_ts / len(fact), 4) if len(fact) else 0.0

findings["invalid_timestamps"] = {
    "null_started_at": int(null_started),
    "null_ended_at": int(null_ended),
    "started_at_gte_ended_at": start_ge_end,
    "outside_program_date_range": out_of_program_range,
    "total_invalid_timestamp_rows": n_invalid_ts,
    "pct_rows_affected": pct_invalid_ts,
    "verdict": verdict(pct_invalid_ts),
}

print(f"NULL started_at              : {null_started:,}")
print(f"NULL ended_at                : {null_ended:,}")
print(f"started_at >= ended_at       : {start_ge_end:,}")
print(f"Before launch / in the future: {out_of_program_range:,}")
print(f"Total invalid-timestamp rows : {n_invalid_ts:,} ({pct_invalid_ts}% of fact_trip)")

**Result: PASS.** 0 null timestamps, 0 trips with `started_at >= ended_at`, and 0 outside the launch-to-now window. The future-date bound is the time of the run, so re-running later moves it.

### Check 4: Negative ride durations

Two independent tests. The first looks for stored durations of zero or less, which the `CHECK` constraint should reject. The second recomputes duration from `ended_at - started_at` and compares it with the stored value, which catches silent derivation bugs that a simple positivity test would miss.

In [ ]:
# Non-positive stored durations (should be impossible given the CHECK constraint)
non_positive_duration = int((fact["duration_minutes"] <= 0).sum())

# Independent recomputation of duration from the raw timestamps
computed_duration = (fact["ended_at"] - fact["started_at"]).dt.total_seconds() / 60.0
# Stored duration disagrees with the recomputed value by more than the tolerance
duration_mismatch = both_present & ((computed_duration - fact["duration_minutes"]).abs() > DURATION_TOLERANCE_MIN)
n_duration_mismatch = int(duration_mismatch.sum())

pct_bad_duration = round(100 * (non_positive_duration + n_duration_mismatch) / len(fact), 4) if len(fact) else 0.0

# Record metrics and verdict for the report
findings["negative_durations"] = {
    "non_positive_duration_rows": non_positive_duration,
    "duration_mismatch_vs_recomputed": n_duration_mismatch,
    "tolerance_minutes": DURATION_TOLERANCE_MIN,
    "pct_rows_affected": pct_bad_duration,
    "verdict": verdict(pct_bad_duration),
}

print(f"duration_minutes <= 0                         : {non_positive_duration:,}")
print(f"duration_minutes vs recomputed mismatch (>{DURATION_TOLERANCE_MIN}min): {n_duration_mismatch:,}")
if non_positive_duration:
    display(fact.loc[fact['duration_minutes'] <= 0, ['ride_id', 'started_at', 'ended_at', 'duration_minutes']].head(10))

**Result: PASS, with a caveat.** 0 non-positive durations, but 314 trips (about 0.002%) have a stored duration that differs from the recomputed value by more than 0.5 minutes. That is under the WARN threshold, yet it points to a small set of inconsistently derived rows. The next step is to inspect those 314 rows. One hypothesis to test is daylight saving: the ETL computed durations from time-zone-naive local timestamps, so a trip spanning a clock change (10 Mar 2024, 3 Nov 2024, 9 Mar 2025, 2 Nov 2025 or 8 Mar 2026) would differ by an hour from the elapsed time the audit recomputes. If the rows cluster on those dates, the mismatch is explained.

### Check 5: Impossible coordinates

Separates two things that look similar but mean different things: coordinates that are **physically impossible** (outside valid latitude/longitude ranges, or the `(0, 0)` "null island" placeholder left by failed geocoding) versus coordinates that are valid but **outside the Chicago service area**, which are legitimate outliers rather than errors. One helper applies the same range test with different bounding boxes.

In [ ]:
def _out_of_range(lat_col, lng_col, lat_range, lng_range):
    """Rows where a non-null coordinate falls outside the given latitude/longitude ranges."""
    lat_bad = fact[lat_col].notna() & ~fact[lat_col].between(*lat_range)
    lng_bad = fact[lng_col].notna() & ~fact[lng_col].between(*lng_range)
    return lat_bad | lng_bad

# Physically impossible coordinates (outside valid latitude/longitude ranges)
impossible_start = _out_of_range("start_lat", "start_lng", WORLD_LAT_RANGE, WORLD_LNG_RANGE)
impossible_end = _out_of_range("end_lat", "end_lng", WORLD_LAT_RANGE, WORLD_LNG_RANGE)
n_impossible = int((impossible_start | impossible_end).sum())

# The (0, 0) placeholder that appears when geocoding silently fails
null_island = (
    ((fact["start_lat"] == 0) & (fact["start_lng"] == 0))
    | ((fact["end_lat"] == 0) & (fact["end_lng"] == 0))
)
n_null_island = int(null_island.sum())

# Valid coordinates outside the Chicago area: an outlier, not an error (start coordinates only)
out_of_service_area = _out_of_range("start_lat", "start_lng", CHI_LAT_RANGE, CHI_LNG_RANGE) & ~impossible_start
n_out_of_service_area = int(out_of_service_area.sum())

# NULL coordinates are reported separately and excluded from the verdict
null_coords = fact[["start_lat", "start_lng", "end_lat", "end_lng"]].isna().any(axis=1).sum()

pct_impossible = round(100 * (n_impossible + n_null_island) / len(fact), 4) if len(fact) else 0.0

# Record metrics and verdict for the report
findings["impossible_coordinates"] = {
    "impossible_lat_lng_rows": n_impossible,
    "null_island_0_0_rows": n_null_island,
    "out_of_chicago_service_area_rows": n_out_of_service_area,
    "any_null_coordinate_rows": int(null_coords),
    "pct_rows_affected": pct_impossible,
    "verdict": verdict(pct_impossible),
}

print(f"Physically impossible lat/lng      : {n_impossible:,}")
print(f"(0, 0) 'null island' coordinates    : {n_null_island:,}")
print(f"Valid but outside Chicago metro area: {n_out_of_service_area:,} (flagged, not an error)")
print(f"Rows with any NULL coordinate       : {null_coords:,}")

**Result: PASS.** 0 impossible coordinates, 0 at (0, 0) and 0 trips starting outside the Chicago box. The one coordinate issue is 17,107 trips (0.104%) with a `NULL` end latitude and longitude. These are reported separately and do not affect the verdict. The service-area test uses start coordinates only.

### Check 6: Null percentages

Runs the same null-rate calculation across every column of every table and surfaces any column above 5% null as an early warning. It feeds the heatmap below.

In [ ]:
# Null rate for every column in every table
null_summary_rows = []
for tname, tdf in tables.items():
    for col in tdf.columns:
        n_null = int(tdf[col].isna().sum())
        null_summary_rows.append({
            "table": tname,
            "column": col,
            "n_rows": len(tdf),
            "n_null": n_null,
            "null_pct": round(100 * n_null / len(tdf), 3) if len(tdf) else 0.0,
        })

# Sorted so the worst columns appear first
null_summary = pd.DataFrame(null_summary_rows).sort_values(["table", "null_pct"], ascending=[True, False])
# Columns above 5% null are surfaced as early warnings
findings["null_percentages"] = {
    "columns_over_5pct_null": null_summary.loc[null_summary["null_pct"] > 5, ["table", "column", "null_pct"]].to_dict("records"),
    "max_null_pct": float(null_summary["null_pct"].max()) if not null_summary.empty else 0.0,
}

# Show only the columns that contain nulls
display(null_summary[null_summary["null_pct"] > 0].reset_index(drop=True))

**Result:** only `fact_trip` contains nulls; all four dimensions are fully populated. Three columns exceed 5%: `is_round_trip` (31.57%), `end_station_key` (20.91%) and `start_station_key` (20.07%). The `is_round_trip` rate is higher than either station column but lower than their sum. At 5,190,701 rows (31.57%) it equals the share of trips missing at least one station, the same figure notebook 05 reports as `missing_station_pct`, so it is `NULL` exactly when a station is missing. It is therefore a downstream symptom of the missing-station problem rather than a separate defect.

### Check 7: Cardinality

Counts distinct values per key column and expresses them as a uniqueness ratio (`n_unique / n_rows`). A ratio near 1 marks an identifier, a ratio near 0 marks a category, and anything unexpected in between deserves a closer look. The same idea is applied to each dimension's natural key.

In [ ]:
# Distinct values per key column: identifiers should be near-unique, categories low-cardinality
cardinality_cols = [
    "ride_id", "start_station_key", "end_station_key", "ride_type_key",
    "member_type_key", "start_hour", "day_of_week", "month_partition",
]
cardinality = pd.DataFrame({
    "n_unique": fact[cardinality_cols].nunique(),
    "n_rows": len(fact),
})
# Uniqueness ratio: near 1 = identifier, near 0 = category
cardinality["uniqueness_ratio"] = (cardinality["n_unique"] / cardinality["n_rows"]).round(4)

# Natural keys of the dimensions should also be unique
dim_cardinality = pd.DataFrame({
    "dim_station.station_id": [dim_station["station_id"].nunique()],
    "dim_ride_type.rideable_type": [dim_ride_type["rideable_type"].nunique()],
    "dim_member_type.member_casual": [dim_member_type["member_casual"].nunique()],
}).T.rename(columns={0: "n_unique"})

# Record for the report
findings["cardinality"] = {
    "fact_trip": cardinality.to_dict("index"),
    "dimensions": dim_cardinality["n_unique"].to_dict(),
}

display(cardinality)
display(dim_cardinality)

**Result:** every column behaves as its role implies. `ride_id` has a ratio of 1.0; the categorical columns are low-cardinality (3 ride types, 2 member types, 24 hours, 7 weekdays, 33 months). 3,918 distinct start stations and 3,953 end stations are in use against 3,957 in `dim_station`, and `dim_station.station_id` has no duplicates.

### Check 8: Data completeness

Two notions of completeness. Column-level completeness covers the fields nearly every analysis needs, averaged into one score. Calendar coverage checks that `dim_date` has every day in its range and that no day has zero rides, which could reveal a service disruption or a month that never loaded.

In [ ]:
# Completeness of the fields nearly every analysis depends on
critical_cols = ["ride_id", "started_at", "ended_at", "start_station_key", "end_station_key", "duration_minutes"]
completeness_by_col = (1 - fact[critical_cols].isna().mean()) * 100
overall_completeness = round(completeness_by_col.mean(), 2)

# Calendar coverage: every day in range should exist in dim_date and have at least one ride
full_calendar = pd.date_range(dim_date["full_date"].min(), dim_date["full_date"].max(), freq="D")
dim_dates_present = set(pd.to_datetime(dim_date["full_date"]).dt.date)
missing_calendar_days = sorted(set(full_calendar.date) - dim_dates_present)

# Days with no rides could indicate an outage or a month missing from the load
ride_days = pd.to_datetime(fact["started_at"], utc=True).dt.date
days_with_no_rides = sorted(set(full_calendar.date) - set(ride_days.dropna()))

# Record for the report
findings["data_completeness"] = {
    "completeness_by_column_pct": completeness_by_col.round(2).to_dict(),
    "overall_completeness_score": overall_completeness,
    "missing_calendar_days_in_dim_date": len(missing_calendar_days),
    "days_with_zero_rides": len(days_with_no_rides),
    "date_range": [str(full_calendar.min().date()), str(full_calendar.max().date())],
}

print("Completeness by critical column:")
print(completeness_by_col.round(2))
print(f"\nOverall completeness score: {overall_completeness}%")
print(f"Missing calendar days in dim_date : {len(missing_calendar_days)} (of {len(full_calendar)} days in range)")
print(f"Days with zero recorded rides      : {len(days_with_no_rides)}")

**Result:** `ride_id`, both timestamps and `duration_minutes` are 100% complete. Start and end station keys are 79.93% and 79.09% complete, which pulls the overall score to **93.17%**; they account for essentially all of the shortfall. There are no calendar gaps and no zero-ride days. One limitation: the range comes from `dim_date`'s own minimum and maximum, so days missing at either end would not be detected.

### Check 9: Data consistency

The most comprehensive check. It re-verifies referential integrity against every dimension, then tests whether the *derived* columns still agree with the raw fields they were computed from: `day_of_week`, `month_partition` and `start_date_key` against `started_at`, and `is_round_trip` against the station keys.

In [ ]:
# Referential integrity against every dimension
valid_date_keys = set(dim_date["date_key"])
valid_ride_type_keys = set(dim_ride_type["ride_type_key"])
valid_member_type_keys = set(dim_member_type["member_type_key"])

orphan_start_date = fact.loc[fact["start_date_key"].notna() & ~fact["start_date_key"].isin(valid_date_keys)]
orphan_end_date = fact.loc[fact["end_date_key"].notna() & ~fact["end_date_key"].isin(valid_date_keys)]
orphan_ride_type = fact.loc[fact["ride_type_key"].notna() & ~fact["ride_type_key"].isin(valid_ride_type_keys)]
orphan_member_type = fact.loc[fact["member_type_key"].notna() & ~fact["member_type_key"].isin(valid_member_type_keys)]

# Derived fields vs. their source: recompute weekday, month and date key from started_at and compare.
# Note: started_at is read in UTC here, but the ETL derived these fields from Chicago local time, so some mismatch is expected (see the result note below)
sub = fact.loc[fact["started_at"].notna() & fact["day_of_week"].notna()].copy()
expected_dow = sub["started_at"].dt.dayofweek + 1
dow_mismatch = int((sub["day_of_week"] != expected_dow).sum())

sub_mp = fact.loc[fact["started_at"].notna() & fact["month_partition"].notna()].copy()
expected_mp = sub_mp["started_at"].dt.year * 100 + sub_mp["started_at"].dt.month
mp_mismatch = int((sub_mp["month_partition"] != expected_mp).sum())

sub_sdk = fact.loc[fact["started_at"].notna() & fact["start_date_key"].notna()].copy()
expected_sdk = sub_sdk["started_at"].dt.strftime("%Y%m%d").astype("int64")
sdk_mismatch = int((sub_sdk["start_date_key"] != expected_sdk).sum())

# is_round_trip vs. the station keys it was derived from
has_both_stations = fact["start_station_key"].notna() & fact["end_station_key"].notna() & fact["is_round_trip"].notna()
expected_round_trip = fact.loc[has_both_stations, "start_station_key"] == fact.loc[has_both_stations, "end_station_key"]
round_trip_mismatch = int((fact.loc[has_both_stations, "is_round_trip"] != expected_round_trip).sum())

# Issues are summed, so a row that fails several tests is counted more than once
n_consistency_issues = int(
    len(orphan_start_date) + len(orphan_end_date) + len(orphan_ride_type) + len(orphan_member_type)
    + dow_mismatch + mp_mismatch + sdk_mismatch + round_trip_mismatch
)
pct_consistency_issues = round(100 * n_consistency_issues / len(fact), 4) if len(fact) else 0.0

# Record for the report
findings["data_consistency"] = {
    "orphan_start_date_refs": int(len(orphan_start_date)),
    "orphan_end_date_refs": int(len(orphan_end_date)),
    "orphan_ride_type_refs": int(len(orphan_ride_type)),
    "orphan_member_type_refs": int(len(orphan_member_type)),
    "day_of_week_mismatches": dow_mismatch,
    "month_partition_mismatches": mp_mismatch,
    "start_date_key_mismatches": sdk_mismatch,
    "is_round_trip_mismatches": round_trip_mismatch,
    "total_consistency_issues": n_consistency_issues,
    "pct_rows_affected": pct_consistency_issues,
    "verdict": verdict(pct_consistency_issues),
}

print(f"Orphan start_date_key / end_date_key refs   : {len(orphan_start_date):,} / {len(orphan_end_date):,}")
print(f"Orphan ride_type_key / member_type_key refs : {len(orphan_ride_type):,} / {len(orphan_member_type):,}")
print(f"day_of_week mismatches vs started_at        : {dow_mismatch:,}")
print(f"month_partition mismatches vs started_at     : {mp_mismatch:,}")
print(f"start_date_key mismatches vs started_at      : {sdk_mismatch:,}")
print(f"is_round_trip mismatches vs station keys     : {round_trip_mismatch:,}")
print(f"Total consistency issues                     : {n_consistency_issues:,} ({pct_consistency_issues}% of fact_trip)")

**Result: FAIL.** Referential integrity is perfect (0 orphaned keys) and `is_round_trip` agrees with the station keys wherever it can be tested. The failures are all in the date-derived columns:
- `day_of_week` and `start_date_key` each disagree with `started_at` in 3,134,008 rows (about 19.1%), an identical count that points to one shared cause.
- `month_partition` disagrees in 100,134 rows (about 0.6%), consistent with trips near a month boundary.

**Likely cause (hypothesis, not yet confirmed): a time-zone mismatch.** This notebook recomputes expected values from `started_at` in UTC, while the ETL derived them from Chicago local time. A trip starting between roughly 7 pm and midnight local time falls on the next day in UTC, which is about 5–6 hours out of 24 and in line with the 19% rate. The same effect applied only to the last hours of a month would explain the much smaller `month_partition` rate. **Corroborating evidence:** notebook 05's coverage query returns `first_start = 2024-01-01 06:00:39+00` and `last_start = 2026-10-01 04:59:38+00`. In Chicago time these are 00:00:39 on 1 January 2024 and 23:59:38 on 30 September 2026, and `dim_date` also ends on 20260930. So `started_at` holds Chicago local time as a `timestamptz`, and reading it in UTC pushes evening trips onto the next calendar day. Recomputing from `started_at.dt.tz_convert("America/Chicago")` would settle it: if the mismatches vanish, the ETL is correct and this check should compare in local time.

**Scoring caveat:** the total adds up *issues*, not distinct rows, so rows failing several tests are counted twice and the 38.73% shown overstates the share of rows affected (closer to 19% if the overlap holds).

## 4. Missing-value visualisation

Two complementary pictures, because a table of numbers hides patterns. The **table × column heatmap** shows at a glance which columns across the whole schema need attention. The **row-level presence matrix** (black = present, white = missing, on a reproducible 500-row sample) answers a different question: do gaps occur independently or cluster together?

In [ ]:
# Table x column grid of null percentages
pivot = null_summary.pivot(index="column", columns="table", values="null_pct")

# Heatmap annotated with exact values
fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(pivot))))
im = ax.imshow(pivot.fillna(-1).values, cmap="Reds", vmin=0, vmax=max(5, pivot.max().max()), aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                     fontsize=8, color="white" if val > pivot.max().max()/2 else "black")
ax.set_title("Null Percentage by Table x Column")
fig.colorbar(im, ax=ax, label="% null")
plt.tight_layout()
# Save the chart alongside the report
plt.savefig(os.path.join(REPORTS_FOLDER, "missing_value_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# Reproducible 500-row sample for readability
sample_n = min(500, len(fact))
sample_cols = [c for c in fact.columns if c not in ("trip_key",)]
sample = fact[sample_cols].sample(n=sample_n, random_state=42).reset_index(drop=True) if sample_n else fact[sample_cols]
# Presence matrix: black = value present, white = missing; shows whether gaps cluster
presence = sample.notna().astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(presence.values, cmap="Greys", aspect="auto", interpolation="nearest")
ax.set_xticks(range(len(presence.columns)))
ax.set_xticklabels(presence.columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks([])
ax.set_ylabel(f"Sampled rows (n={sample_n})")
ax.set_title("Row-Level Missing Value Matrix — fact_trip sample (black = present, white = missing)")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_FOLDER, "missing_value_matrix_rows.png"), dpi=150)
plt.show()

**Finding:** missingness is not independent. `is_round_trip` is missing more often than either station key alone, and `end_lat` and `end_lng` are missing together in the same 17,107 rows. That is the signature of a shared root cause, trips with no recorded station, rather than several unrelated defects. The 17,107 trips with null end coordinates (0.10%) are a separate and much smaller issue: `end_lat` and `end_lng` always go missing together, because the any-null-coordinate count equals each column's null count.

## 5. Composite data-quality score

Nine checks produce nine numbers, which is useful for debugging but hard to track. This section rolls them into five standard dimensions, each scored `100 - % of rows affected`, and combines them into one weighted score with a letter grade.

| Dimension | Weight | Driven by |
|---|---|---|
| Uniqueness | 20% | duplicate `ride_id` rate |
| Completeness | 20% | critical-column null rate |
| Validity | 25% | invalid timestamps, duration errors, impossible coordinates |
| Consistency | 20% | orphaned keys and derived-field mismatches |
| Accuracy | 15% | `is_anomalous` rate and calendar gaps |

Validity has the highest weight by design: a warehouse tolerates a few gaps better than physically invalid data reaching an analysis.

In [ ]:
def score_from_pct(pct_affected: float) -> float:
    """Convert a percentage of affected rows into a 0-100 score (floored at 0)."""
    return max(0.0, 100.0 - pct_affected)

n = len(fact)

# Five quality dimensions, each scored as 100 minus the percentage of rows affected
uniqueness_score = score_from_pct(findings["duplicate_ride_ids"]["pct_rows_affected"])
completeness_score = findings["data_completeness"]["overall_completeness_score"]

# Validity pools timestamp, duration and coordinate errors
validity_bad_rows = (
    findings["invalid_timestamps"]["total_invalid_timestamp_rows"]
    + findings["negative_durations"]["non_positive_duration_rows"]
    + findings["negative_durations"]["duration_mismatch_vs_recomputed"]
    + findings["impossible_coordinates"]["impossible_lat_lng_rows"]
    + findings["impossible_coordinates"]["null_island_0_0_rows"]
)
validity_pct = round(100 * validity_bad_rows / n, 4) if n else 0.0
validity_score = score_from_pct(validity_pct)

consistency_score = score_from_pct(findings["data_consistency"]["pct_rows_affected"])

# Accuracy reflects the ETL anomaly rate plus any calendar gaps (it relies on the ETL's own flag)
anomalous_rate_pct = round(100 * fact["is_anomalous"].fillna(False).mean(), 4) if "is_anomalous" in fact else 0.0
calendar_gap_pct = round(
    100 * findings["data_completeness"]["missing_calendar_days_in_dim_date"]
    / max(1, (pd.to_datetime(findings["data_completeness"]["date_range"][1])
              - pd.to_datetime(findings["data_completeness"]["date_range"][0])).days + 1),
    4,
)
accuracy_score = score_from_pct(anomalous_rate_pct + calendar_gap_pct)

dimension_scores = {
    "Uniqueness": round(uniqueness_score, 2),
    "Completeness": round(completeness_score, 2),
    "Validity": round(validity_score, 2),
    "Consistency": round(consistency_score, 2),
    "Accuracy": round(accuracy_score, 2),
}
# Validity carries the highest weight (25%): invalid data is costlier than a few gaps
weights = {"Uniqueness": 0.20, "Completeness": 0.20, "Validity": 0.25, "Consistency": 0.20, "Accuracy": 0.15}

# Weighted composite score
overall_score = round(sum(dimension_scores[k] * weights[k] for k in weights), 2)

def grade_for(score):
    """Map a 0-100 score to a letter grade."""
    if score >= 90: return "A"
    if score >= 80: return "B"
    if score >= 70: return "C"
    if score >= 60: return "D"
    return "F"

overall_grade = grade_for(overall_score)

# Record for the report
findings["quality_score"] = {
    "dimension_scores": dimension_scores,
    "weights": weights,
    "overall_score": overall_score,
    "overall_grade": overall_grade,
}

score_df = pd.DataFrame({
    "dimension": list(dimension_scores.keys()),
    "score": list(dimension_scores.values()),
    "weight": [weights[k] for k in dimension_scores],
})
display(score_df)
print(f"\nOVERALL DATA QUALITY SCORE: {overall_score} / 100  (Grade {overall_grade})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# Green at 90 and above, amber at 70 and above, red below
colors = ["#2e7d32" if s >= 90 else "#f9a825" if s >= 70 else "#c62828" for s in dimension_scores.values()]
bars = ax.bar(dimension_scores.keys(), dimension_scores.values(), color=colors)
ax.set_ylim(0, 100)
ax.set_ylabel("Score (0-100)")
ax.set_title(f"Data Quality Dimension Scores — Overall: {overall_score} (Grade {overall_grade})")
ax.axhline(90, color="grey", linestyle="--", linewidth=0.8)
for b, s in zip(bars, dimension_scores.values()):
    ax.text(b.get_x() + b.get_width()/2, s + 1.5, f"{s:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_FOLDER, "quality_score_breakdown.png"), dpi=150)
plt.show()

**Result:** Uniqueness 100.00 · Completeness 93.17 · Validity 100.00 · Consistency 61.27 · Accuracy 97.43, giving **90.5 / 100, Grade A**. Consistency is the only dimension below 90 and is drawn in red in the chart.

**How to read the score**
- The A is borderline (90.5 against a cut-off of 90) and coexists with two FAIL verdicts, so it should be read alongside the per-check results.
- Consistency is probably understated because issue counts are summed. If only about 19% of rows were affected, Consistency would be about 81 and the overall score about 94.
- Validity excludes the 17,107 null end coordinates, and Accuracy relies on the ETL's own `is_anomalous` flag, so it is not a fully independent measure.

## 6. Data-quality report

All findings are compiled into one report and rendered in the notebook as Markdown. The same `findings` dictionary is then persisted three ways: Markdown for people, JSON for scripts and pipelines, and a flattened one-metric-per-row CSV for spreadsheets and BI tools such as Power BI. Producing every format from one source means the numbers cannot drift apart.

Checks 6–8 have no verdict, so they appear in their own report sections rather than in the check-results table; this is why that table's numbering jumps from 5 to 9.

In [ ]:
def build_markdown_report() -> str:
    """Assemble the data-quality report as Markdown from the shared findings."""
    lines = []
    lines.append(f"# Divvy Data Quality Report")
    lines.append(f"**Run timestamp (UTC):** {RUN_TIMESTAMP}  ")
    lines.append(f"**Rows assessed (fact_trip):** {len(fact):,}  ")
    lines.append(f"**Overall Quality Score:** {overall_score} / 100 (Grade {overall_grade})\n")

    lines.append("## Table Row Counts")
    for tname, tdf in tables.items():
        lines.append(f"- `{tname}`: {len(tdf):,} rows")
    lines.append("")

    lines.append("## Check Results\n")
    lines.append("| # | Check | Key Metric | % Rows Affected | Verdict |")
    lines.append("|---|---|---|---|---|")
    duplicate_ids = findings["duplicate_ride_ids"]

    duplicate_ids = findings["duplicate_ride_ids"]
    stations = findings["missing_stations"]
    timestamps = findings["invalid_timestamps"]
    durations = findings["negative_durations"]
    coordinates = findings["impossible_coordinates"]
    consistency = findings["data_consistency"]

    # Checks 6-8 (nulls, cardinality, completeness) report raw metrics without a verdict, so they appear in their own sections
    check_rows = [
    	(
            1,
            "Duplicate ride IDs",
            f"{duplicate_ids['duplicate_rows_total']:,} duplicate rows",
            duplicate_ids["pct_rows_affected"],
            duplicate_ids["verdict"],
        ),
        (
            2,
            "Missing stations",
            (
                f"{stations['null_start_station_key'] + stations['null_end_station_key']:,} "
                "null station refs"
            ),
            stations["pct_rows_affected"],
            stations["verdict"],
        ),
        (
            3,
            "Invalid timestamps",
            f"{timestamps['total_invalid_timestamp_rows']:,} invalid rows",
            timestamps["pct_rows_affected"],
            timestamps["verdict"],
        ),
        (
            4,
            "Negative ride durations",
            (
                f"{durations['non_positive_duration_rows']:,} non-positive + "
                f"{durations['duration_mismatch_vs_recomputed']:,} mismatched"
            ),
            durations["pct_rows_affected"],
            durations["verdict"],
        ),
        (
            5,
            "Impossible coordinates",
            (
                f"{coordinates['impossible_lat_lng_rows']:,} impossible + "
                f"{coordinates['null_island_0_0_rows']:,} (0,0)"
            ),
            coordinates["pct_rows_affected"],
            coordinates["verdict"],
        ),
        (
            9,
            "Data consistency",
            f"{consistency['total_consistency_issues']:,} issues",
            consistency["pct_rows_affected"],
            consistency["verdict"],
        ),
    ]

    for n_, check, metric, pct, verd in check_rows:
        lines.append(f"| {n_} | {check} | {metric} | {pct}% | {verd} |")
    lines.append("")

    lines.append("## Null Percentages (columns with any nulls)\n")
    lines.append(null_summary[null_summary['null_pct'] > 0].to_markdown(index=False))
    lines.append("")

    lines.append("## Cardinality (fact_trip)\n")
    lines.append(cardinality.to_markdown())
    lines.append("")

    lines.append("## Data Completeness\n")
    lines.append(f"- Overall completeness score: **{findings['data_completeness']['overall_completeness_score']}%**")
    lines.append(f"- Missing calendar days in `dim_date`: {findings['data_completeness']['missing_calendar_days_in_dim_date']}")
    lines.append(f"- Days with zero recorded rides: {findings['data_completeness']['days_with_zero_rides']}")
    lines.append(f"- Date range assessed: {findings['data_completeness']['date_range'][0]} to {findings['data_completeness']['date_range'][1]}")
    lines.append("")

    lines.append("## Data Quality Score Breakdown\n")
    lines.append(score_df.to_markdown(index=False))
    lines.append("")
    lines.append(f"**Overall: {overall_score} / 100 — Grade {overall_grade}**")

    return "\n".join(lines)

# Render the report in the notebook
report_md = build_markdown_report()
display(Markdown(report_md))

In [ ]:
# Persist three formats from one source of truth so the numbers always agree
md_path = os.path.join(REPORTS_FOLDER, "data_quality_report.md")
json_path = os.path.join(REPORTS_FOLDER, "data_quality_report.json")
csv_path = os.path.join(REPORTS_FOLDER, "data_quality_metrics.csv")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(report_md)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2, default=str)

# Flatten scalar metrics to one row per metric for spreadsheet and BI import
flat_rows = []
for check, metrics in findings.items():
    if not isinstance(metrics, dict):
        continue
    for k, v in metrics.items():
        if isinstance(v, (dict, list)):
            continue
        flat_rows.append({"check": check, "metric": k, "value": v})
pd.DataFrame(flat_rows).to_csv(csv_path, index=False)

print(f"Markdown report : {md_path}")
print(f"JSON report     : {json_path}")
print(f"CSV metrics     : {csv_path}")
print(f"Charts          : {REPORTS_FOLDER}")

## Summary

### Purpose
Notebook 04 is an independent audit of the loaded warehouse. Instead of trusting the ETL's own counters, it reads all five `divvy` tables once (a consistent snapshot of 16,440,836 trips) and re-derives data-quality metrics from the data itself.

### What was built
- **Nine checks:** duplicate IDs, missing stations, invalid timestamps, negative durations, impossible coordinates, null percentages, cardinality, completeness and consistency.
- **Configurable verdict rules:** for checks 1–5 and 9, at least 1% of rows affected is FAIL, at least 0.1% is WARN, otherwise PASS.
- **Missingness visuals:** a table-by-column null heatmap and a row-level presence matrix.
- **Composite score:** five weighted dimensions combined into a 0–100 score with a letter grade.
- **Multi-format reports:** Markdown, JSON and a flattened metrics CSV, generated from one shared dictionary.

### Key results (run of 2026-10-16)

| # | Check | Result | Verdict |
|---|---|---|---|
| 1 | Duplicate ride IDs | 0 | PASS |
| 2 | Missing stations | 20.07% null start key, 20.91% null end key; 0 orphaned keys | **FAIL** |
| 3 | Invalid timestamps | 0 | PASS |
| 4 | Negative durations | 0 non-positive; 314 mismatches vs. recomputed (about 0.002%) | PASS |
| 5 | Impossible coordinates | 0 impossible; 17,107 null end coordinates reported separately | PASS |
| 6–8 | Nulls, cardinality, completeness | `is_round_trip` 31.57% null; all columns behave as expected; 93.17% completeness | no verdict |
| 9 | Consistency | 0 orphaned keys; `day_of_week` and `start_date_key` mismatch in about 19.1% of rows; `month_partition` in about 0.6% | **FAIL** |

**Composite score: 90.5 / 100, Grade A** (Uniqueness 100, Validity 100, Accuracy 97.43, Completeness 93.17, Consistency 61.27).

### Interpretation
- **Missing stations** are a completeness problem rather than an integrity problem. The likely cause, not yet confirmed, is rides without a docking station such as dockless or e-bike rides. It also explains the `is_round_trip` nulls. Station-level analysis therefore covers only about two-thirds of trips, since 31.57% lack at least one station.
- **Date-field mismatches** are probably a time-zone artifact: the audit recomputes values from UTC timestamps while the ETL derived them from Chicago local time. Notebook 05's `first_start` (06:00:39 UTC) and `last_start` (04:59:38 UTC) equal 00:00:39 and 23:59:38 Chicago time, which fits. The confirming recompute in `America/Chicago` has not been run yet.
- **Reading the grade:** the A is borderline and coexists with two FAIL verdicts, so it must be read with the per-check results. Consistency is likely understated because issue counts are summed; at about 19% of rows affected, the overall score would be roughly 94.

### Recommended next steps
1. Recompute the day, date-key and month checks in `America/Chicago` time to decide whether the mismatches are an ETL bug or an audit artifact.
2. Cross-tabulate null stations against ride type.
3. Inspect the 314 duration mismatches (do they cluster on the five daylight-saving changes in the window?) and the 17,107 null end coordinates.
4. Count consistency issues by distinct rows instead of summing issue counts.
5. Re-run after every load so the score becomes a trend line.

### Hand-off
Notebook 05 uses these findings as its basis: ETL-flagged anomalies are excluded and station metrics are treated with caution.

## Skills Demonstrated

**Data quality and validation**
- Independent data-quality auditing of a 16.4-million-row warehouse, re-deriving metrics instead of trusting upstream counters
- Uniqueness, referential-integrity, timestamp, range, geospatial, null, cardinality, completeness and consistency testing
- Cross-validating derived and engineered fields against their source columns
- Distinguishing genuine errors from valid-but-unusual outliers
- Decomposing one failure into separately diagnosable causes (NULL keys, orphaned keys, incomplete dimension rows)

**Analytics, scoring and statistics**
- Composite KPI and scorecard design: weighted multi-dimension scoring with a letter grade
- Auditable PASS / WARN / FAIL thresholds defined in configuration
- Hypothesis-driven root-cause analysis (UTC versus local time, dockless rides) with explicit caveats
- Critical reading of the score itself: borderline grade, double-counting, exclusions and dependence on an upstream flag

**Python, SQL and visualization**
- pandas and NumPy on millions of rows from a single in-memory snapshot
- SQL access with `psycopg2` and proper connection lifecycle management
- Data visualization: null-percentage heatmap, `missingno`-style presence matrix and a colour-coded score chart
- Reusable helpers (`run_query`, `verdict`, `score_from_pct`) and shared-state design that keeps checks composable

**Reporting and communication**
- Automated multi-format deliverables (Markdown for people, JSON for scripts, CSV for BI tools) from one source of truth
- Translating technical findings into a business-readable report with prioritized next steps
- Framing data quality as ongoing monitoring (a trend line) rather than a one-off audit